In [3]:
import os
import time
import pandas as pd
from ytmusicapi import YTMusic

In [33]:

class YTMusicPlaylists:

    def __init__(self, header='../headers_auth.json', playlist_limit=2000):
        self.yt = YTMusic(header)
        self.playlist_limit = playlist_limit
        self.playlists = pd.DataFrame(self.yt.get_library_playlists(limit=playlist_limit))

    def _playlist_loc_first(self, col, value):
        res = self.playlists.loc[self.playlists[col] == value]
        if len(res) == 0:
            print(f'No playlist with {col}: {value}')
        elif len(res) > 1:
            print(f'multiple matches for : {value}, choosing first result of:\n {res}')  
        return res.iloc[0]

    def query_by_title(self, title):
        return  self._playlist_loc_first(col='title', value=title)

    def query_by_playlistId(self, playlistId):
        return  self._playlist_loc_first(col='playlistId', value=playlistId)

    def playlist_get_info(self, playlistId, playlist_limit=None):
        if not playlist_limit:
            playlist_limit = self.playlist_limit
        return self.yt.get_playlist(playlistId, limit=playlist_limit)

    def playlist_from_tsv(self, tsv_path):
        assert tsv_path.enswith('.tsv')
        df = pd.read_csv(tsv_path, sep='\t', index_col=0)
        pl_name = os.path.basename(tsv_path).split('.tsv')[0]
        print(f'\nGenerating {pl_name} ytmusic playlist for {len(df)} tracks')
        vids = df.videoId.unique().tolist()
        desc = f'Matched {len(vids)} tracks from {pl_name} manually uploaded from local tsv.'
        pl_id = self.yt.create_playlist(title=pl_name,  description=desc, privacy_status='PRIVATE', video_ids=list(vids))
        print(f'Saved {len(vids)} {pl_name} tracks playlist with id: {pl_id}')

    def playlist_rate_all_songs(self, playlistId, rating, sleep_time=0.5, verbose=False):
        assert rating in ('LIKE', 'DISLIKE', 'UNRATED')
        playlist = self.query_by_playlistId(playlistId)
        playlist_meta = self.playlist_get_info(playlistId)
        num_tracks = len(playlist_meta["tracks"])
        if verbose: 
            print(f'Found {num_tracks} tracks to rate as {rating} in playlist: {playlist["title"]} ({playlistId})')
        rate_count = 0
        for track in playlist_meta["tracks"]:
            if track["likeStatus"] == rating:
                continue
            if verbose: 
                print(f'Setting rating for {track["videoId"]} to {rating}')
            self.yt.rate_song(track["videoId"], rating=rating)
            rate_count += 1
            time.sleep(sleep_time) 
        print(f'Rated {rate_count} of {num_tracks} tracks as {rating} in playlist {playlist["title"]} ({playlistId})')

    def _playlist_is_dislike(self, name):
        name = name.lower()
        if 'not like' in name:
            return True
        if ' dislike' in name:
            return True
        if 'thumbs_down' in name:
            return True

    def _playlist_is_like(self, name):
        name = name.lower()
        if self._playlist_is_dislike(name):
            return False
        if 'thumbs_up' in name:
            return True
        if ' like' in name or ' likes' in name:
            return True
        if ' top' in name:
            return True
            
    def playlist_get_all_like_playlists(self):
        like_playlists_ids = {}
        for i, row in self.playlists.iterrows():
            if self._playlist_is_like(row.title.lower()):
                like_playlists_ids[row.title] = row.playlistId
        return like_playlists_ids

In [35]:
Y = YTMusicPlaylists(header='../headers_auth.json')

## Rate playlists that should have all LIKE as LIKE

In [38]:
MAX_PLAYLIST_SIZE_TO_RATE = 500
for title, playlistId in Y.playlist_get_all_like_playlists().items():
    print(100*'=')
    info = Y.playlist_get_info(playlistId, playlist_limit=MAX_PLAYLIST_SIZE_TO_RATE)
    num_tracks = len(info['tracks'])
    if num_tracks >= MAX_PLAYLIST_SIZE_TO_RATE:
        print(f'Skipping playlist: {title} ({playlistId}) which has {MAX_PLAYLIST_SIZE_TO_RATE} or more tracks')
        continue
    print(f'Playlist: {title} ({playlistId}) has {num_tracks} tracks')
    Y.playlist_rate_all_songs(playlistId, rating='LIKE')

Skipping playlist: Your Likes (LM) which has 500 or more tracks
Skipping playlist: zz__thumbs_up (PLWptjpDqazOxbzZ60EYhRT0XtAr6nh_xH) which has 500 or more tracks
Playlist: z_dj_thumbs_up (PLWptjpDqazOwqMTsUps2PvNbVRUP3BJnU) has 336 tracks
Found 336 tracks to rate as LIKE in playlist: z_dj_thumbs_up (PLWptjpDqazOwqMTsUps2PvNbVRUP3BJnU)
Rated 13 of 336 tracks as LIKE in playlist z_dj_thumbs_up (PLWptjpDqazOwqMTsUps2PvNbVRUP3BJnU)
Playlist: y_2020_thumbs_up (PLWptjpDqazOwKZr6jtEgCCuLFDumYsJ6-) has 14 tracks
Found 14 tracks to rate as LIKE in playlist: y_2020_thumbs_up (PLWptjpDqazOwKZr6jtEgCCuLFDumYsJ6-)
Rated 0 of 14 tracks as LIKE in playlist y_2020_thumbs_up (PLWptjpDqazOwKZr6jtEgCCuLFDumYsJ6-)
Skipping playlist: y_2019_thumbs_up (PLWptjpDqazOx4Du7ehdzqeASga8HCvMo2) which has 500 or more tracks
Skipping playlist: y_2018_thumbs_up (PLWptjpDqazOyfctX2Yl0FFyHicZYyY-sS) which has 500 or more tracks
Skipping playlist: y_2017_thumbs_up (PLWptjpDqazOzq39_MwaIsAr3ezFMJIrO_) which has 500 or m

In [17]:
Y.playlists.iloc[30]

title                                          y_2005s_thumbs_up
playlistId                    PLWptjpDqazOw4qhQM-E7Q3gESj9vHxW3N
thumbnails     [{'url': 'https://yt3.ggpht.com/JAAqBvu8IciwsO...
description                                 Jake G • 1,417 songs
count                                                      1,417
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
Name: 30, dtype: object

In [18]:
r = Y.query_by_title('y_2005s_thumbs_up')